# YOLO Bukukia Training Pipeline — v2.3 (GCP VM)

This notebook is for execution on a **GCP VM instance** (not Google Colab).
Run cells sequentially in JupyterLab or VS Code on the VM.

## Differences from Colab version
- No Google Drive mount
- Project files are on the VM's local disk
- GPU accessed directly via CUDA
- System packages installed via `apt` if needed

## Pipeline Overview

| Step | Section | Script |
|---|---|---|
| 0 | System Check + Setup | — |
| 1 | Configuration | — |
| 2 | Training (Focal Loss) | `train_2.py` |
| 3 | Model Evaluation | `test_model.py` |
| 4 | Fine-Tuning *(optional)* | `fine_tune.py` |
| 5 | Manual / Advanced Tools | various |

---
## 0. System Check & Setup

Run this once to verify GPU availability and install required packages.

In [ ]:
# ── GPU Check ──
!nvidia-smi

In [ ]:
# ── Install system dependencies (run once) ──
!apt-get install -y libgl1-mesa-glx libglib2.0-0 -q

# ── Install Python dependencies ──
!pip install ultralytics>=8.3.0 opencv-python-headless matplotlib pandas -q

print("Done.")

In [ ]:
import os, sys
from pathlib import Path

# ── Set project directory ──
# Change this to where the project is cloned/uploaded on your VM
PROJECT_DIR = Path('')

if not PROJECT_DIR.exists():
    raise FileNotFoundError(f"Project directory not found: {PROJECT_DIR}\n"
                            f"Please update PROJECT_DIR to your actual path.")

os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR / 'scripts'))
print(f"Working dir : {os.getcwd()}")
print(f"Python      : {sys.executable}")

# Verify key scripts exist
scripts = ['train_2.py', 'test_model.py', 'config.py', 'iou_callback.py',
           'evaluate_iou_full.py', 'fine_tune.py']
for s in scripts:
    path = PROJECT_DIR / 'scripts' / s
    status = '✅' if path.exists() else '❌ MISSING'
    print(f"  scripts/{s}: {status}")

In [ ]:
# ── Verify config and print environment ──
from scripts.config import print_config
print_config()

---
## 1. Configuration

Set model and training parameters via environment variables (read by `config.py`).

| Parameter | Description | Default |
|---|---|---|
| `YOLO_MODEL` | Model checkpoint name or path | `yolo26n.pt` |
| `YOLO_EPOCHS` | Number of training epochs | `50` |
| `YOLO_CONF` | Confidence threshold for predictions | `0.25` |

In [ ]:
import os

# ── Model & Training Settings ──
os.environ['YOLO_MODEL']  = 'yolo26n.pt'
os.environ['YOLO_EPOCHS'] = '50'
os.environ['YOLO_CONF']   = '0.25'

print(f"Model  : {os.environ['YOLO_MODEL']}")
print(f"Epochs : {os.environ['YOLO_EPOCHS']}")
print(f"Conf   : {os.environ['YOLO_CONF']}")

---
## 2. Training — `train_2.py` (Focal Loss)

Trains the YOLO model with **Focal Loss** support and **IoU callback** tracking.
Includes dataset preparation, 80/20 train/val split, data.yaml creation, training, and post-training analysis.

| | Path / Description |
|---|---|
| **Input** | `input_files/raw_images/` — Training images |
| **Input** | `input_files/export/labels/*.txt` — YOLO labels (normalized 0–1) |
| **Input** | `input_files/export/classes.txt` — Class names |
| **Output** | `results/runs/train/weights/best.pt` — Best trained model |
| **Output** | `results/runs/train/training_analysis.png` — Training dashboard |
| **Output** | `results/runs/train/iou_log.csv` — Per-epoch Mean IoU log |

### Arguments

| Argument | Type | Default | Description |
|---|---|---|---|
| `--model` | str | config | Model name or path (e.g. `yolo26n.pt`, `yolo26m.pt`) |
| `--epochs` | int | config | Number of training epochs |
| `--imgsz` | int | `640` | Image size for training |
| `--batch` | int | `-1` | Batch size (`-1` = auto) |
| `--limit` | int | `0` | Limit total images (0 = use all) |
| `--fraction` | float | `0.0` | Use % of data (e.g. `0.2` = 20%; overrides `--limit`) |
| `--images-dir` | str | `raw_images` | Custom image folder path |
| `--split-dir` | str | — | Pre-split dataset directory (skips auto-split) |
| `--fl-gamma` | float | `1.5` | Focal Loss gamma (0.0 = disable, use standard BCE) |
| `--fl-alpha` | float | `0.25` | Focal Loss alpha (class-balance factor) |

> 💡 **Focal Loss**: Set `--fl-gamma 0.0` to disable and use standard BCE loss.
> Use `--split-dir` to skip the automatic 80/20 split and provide your own pre-split dataset.

In [ ]:
# ── Training with Focal Loss ──
# Adjust arguments as needed:

!python scripts/train_2.py \
    --model $YOLO_MODEL \
    --epochs $YOLO_EPOCHS \
    --imgsz 640 \
    --batch -1 \
    --fl-gamma 1.5 \
    --fl-alpha 0.25

### Alternative: Train with Pre-Split Dataset

If you already have a pre-split dataset (e.g. `dataset_fixpage96`), use `--split-dir` to skip auto-split.

In [ ]:
# ── Alternative: Train with pre-split dataset ──
# Uncomment and adjust the --split-dir path:

# !python scripts/train_2.py \
#     --model $YOLO_MODEL \
#     --epochs $YOLO_EPOCHS \
#     --imgsz 640 \
#     --batch -1 \
#     --fl-gamma 1.5 \
#     --fl-alpha 0.25 \
#     --split-dir input_files/dataset_fixpage96

---
## 3. Model Evaluation — `test_model.py`

Evaluate the trained model on a separate test set (not used in training).

| | Path / Description |
|---|---|
| **Input** | `results/runs/train/weights/best.pt` — Trained model |
| **Input** | `input_files/test-dataset/` — Test images (with optional ground truth labels) |
| **Output** | `results/test_results/test_dashboard.png` — Visual dashboard |
| **Output** | `results/test_results/annotated/*.jpg` — Annotated detection images |
| **Output** | `results/test_results/all_results_grid.png` — Grid of all results |
| **Output** | `results/test_results/test_report.txt` — Metrics summary |
| **Output** | `results/test_results/iou_distribution.png` — IoU histogram (if GT available) |

### Arguments

| Argument | Type | Default | Description |
|---|---|---|---|
| `--conf` | float | `0.25` | Confidence threshold |
| `--model` | str | auto-detected | Path to a specific model file (.pt) |
| `--data` | str | `input_files/test-dataset` | Path to test images directory |

In [ ]:
# ── Model Evaluation ──

!python scripts/test_model.py \
    --conf  $YOLO_CONF \
    --model results/runs/train/weights/best.pt \
    --data  input_files/test-dataset

---
## 4. Fine-Tuning *(Optional)* — `fine_tune.py`

Uses Ultralytics built-in `model.tune()` for hyperparameter search.

| | Path / Description |
|---|---|
| **Input** | `results/runs/train/weights/best.pt` — Best model from Step 2 |
| **Input** | `results/dataset/data.yaml` — Dataset config (auto-created by training) |
| **Output** | `results/runs/fine_tune/` — Per-trial results |
| **Output** | `results/runs/fine_tune/best_hyperparameters.yaml` — Best hyperparameters |

### Arguments

| Argument | Type | Default | Description |
|---|---|---|---|
| `--model` | str | auto-detected | Path to the model .pt to fine-tune |
| `--iterations` | int | `30` | Number of tuning iterations |
| `--epochs` | int | `30` | Epochs per tuning trial |
| `--imgsz` | int | `640` | Image size |

In [ ]:
# ── Fine-Tuning (Optional) ──
# Uncomment to run:

# !python scripts/fine_tune.py \
#     --model results/runs/train/weights/best.pt \
#     --iterations 30 \
#     --epochs 30 \
#     --imgsz 640

---
---
# 5. Manual / Advanced Tools

Standalone utility scripts for evaluation, data export, and data management.
Each section below is independent and ready to run.

---
### 5.1 Evaluate IoU — Validation Set (`evaluate_iou_full.py`)

Full IoU evaluation on the validation dataset with per-class box plots and CSV export.

| Argument | Type | Default | Description |
|---|---|---|---|
| `--model` | str | **required** | Path to trained model (.pt) |
| `--data` | str | **required** | Validation dataset directory (with `images/val` and `labels/val`) |
| `--out` | str | `.` | Output directory for CSV and PNG results |
| `--conf` | float | `0.25` | Confidence threshold |
| `--iou-thres` | float | `0.5` | IoU threshold for matching |

In [ ]:
!python scripts/evaluate_iou_full.py \
    --model results/runs/train/weights/best.pt \
    --data  results/dataset \
    --out   results/iou_results

---
### 5.2 Evaluate IoU — Training Set (`evaluate_iou_full_train.py`)

Same IoU evaluation on the **training** set — useful for checking overfitting.

| Argument | Type | Default | Description |
|---|---|---|---|
| `--model` | str | **required** | Path to trained model (.pt) |
| `--data` | str | **required** | Dataset directory (with `images/train` and `labels/train`) |
| `--out` | str | `.` | Output directory for CSV and PNG results |
| `--conf` | float | `0.25` | Confidence threshold |
| `--iou-thres` | float | `0.5` | IoU threshold for matching |

In [ ]:
!python scripts/evaluate_iou_full_train.py \
    --model results/runs/train/weights/best.pt \
    --data  results/dataset \
    --out   results/iou_results_train

---
### 5.3 Predict to Labels (`predict_to_labels.py`)

Run YOLO predictions on images and save results as YOLO-format label `.txt` files.

| Argument | Type | Default | Description |
|---|---|---|---|
| `--model` | str | **required** | Path to trained model (.pt) |
| `--source` | str | **required** | Directory of images to predict on |
| `--out` | str | `predicted_labels` | Output directory for label .txt files |
| `--conf` | float | `0.25` | Confidence threshold |
| `--imgsz` | int | `640` | Image size for inference |

In [ ]:
!python scripts/predict_to_labels.py \
    --model  results/runs/train/weights/best.pt \
    --source input_files/raw_images \
    --out    results/predicted_labels \
    --conf   0.25

---
### 5.4 Extract Label Details (`extract_label_details.py`)

Extract all label annotations from `.txt` files into a CSV with class names and bounding box info.

| Argument | Type | Default | Description |
|---|---|---|---|
| `--labels-dir` | str | **required** | Directory containing YOLO label .txt files |
| `--classes-txt` | str | **required** | Path to `classes.txt` |
| `--out` | str | `label_details.csv` | Output CSV file path |

In [ ]:
!python scripts/extract_label_details.py \
    --labels-dir  input_files/export/labels \
    --classes-txt input_files/export/classes.txt \
    --out         results/label_details.csv

---
### 5.5 Manual Hyperparameter Tuning (`fine_tune_tuningmanual.py`)

Coordinate-descent hyperparameter sweep with heatmap/scatter plot visualization.
More control than `fine_tune.py` — you specify exact parameter ranges.

| Argument | Type | Default | Description |
|---|---|---|---|
| `--model` | str | auto-detected | Path to model .pt |
| `--epochs` | int | `30` | Epochs per trial |
| `--imgsz` | int | `640` | Image size |
| `--batch` | int | `16` | Batch size |
| `--rounds` | int | `2` | Number of coordinate descent rounds |

In [ ]:
# Uncomment to run:

# !python scripts/fine_tune_tuningmanual.py \
#     --model results/runs/train/weights/best.pt \
#     --epochs 30 \
#     --imgsz 640 \
#     --batch 16 \
#     --rounds 2

---
### 5.6 Copy Images from Labels (`copy_images_from_labels_colab.py`)

Copy image files whose filenames match existing label file names.

| Argument | Type | Default | Description |
|---|---|---|---|
| `--labels-dir` | str | **required** | Directory of label .txt files |
| `--images-dir` | str | **required** | Source directory of images |
| `--out` | str | **required** | Destination directory for matched images |

In [ ]:
!python scripts/copy_images_from_labels_colab.py \
    --labels-dir input_files/export/labels \
    --images-dir input_files/raw_images \
    --out        input_files/matched_images

---
### 5.7 Filter Images by CSV (`filter_images_colab.py`)

Copy images whose filenames appear in a CSV file (column `filename`).

| Argument | Type | Default | Description |
|---|---|---|---|
| `--csv` | str | **required** | CSV file with a `filename` column |
| `--images-dir` | str | **required** | Source images directory |
| `--out` | str | **required** | Destination directory for filtered images |

In [ ]:
!python scripts/filter_images_colab.py \
    --csv        results/filter_list.csv \
    --images-dir input_files/raw_images \
    --out        input_files/filtered_images

---
### 5.8 Filter Labels by CSV (`filter_labels_colab.py`)

Copy label files whose filenames appear in a CSV file (column `filename`).

| Argument | Type | Default | Description |
|---|---|---|---|
| `--csv` | str | **required** | CSV file with a `filename` column |
| `--labels-dir` | str | **required** | Source labels directory |
| `--out` | str | **required** | Destination directory for filtered labels |

In [ ]:
!python scripts/filter_labels_colab.py \
    --csv        results/filter_list.csv \
    --labels-dir input_files/export/labels \
    --out        input_files/filtered_labels

---
### 5.9 Filter Labels by Segment Class (`filter_labels_by_segment_colab.py`)

Filter label files to keep only rows matching specific segment class IDs.

| Argument | Type | Default | Description |
|---|---|---|---|
| `--labels-dir` | str | **required** | Source labels directory |
| `--out` | str | **required** | Destination directory for filtered labels |
| `--classes` | int[] | **required** | List of class IDs to keep (space-separated) |

In [ ]:
!python scripts/filter_labels_by_segment_colab.py \
    --labels-dir input_files/export/labels \
    --out        input_files/filtered_by_segment \
    --classes 0 1 2

---
### 5.10 Move/Copy Random Images (`move_random_images_colab.py`)

Randomly select N images (and matching labels) and copy or move them to a target directory.

| Argument | Type | Default | Description |
|---|---|---|---|
| `--images-dir` | str | **required** | Source images directory |
| `--labels-dir` | str | — | Source labels directory (to copy matching labels) |
| `--out` | str | **required** | Destination directory |
| `--count` | int | **required** | Number of images to select |
| `--move` | flag | `False` | Move instead of copy |
| `--seed` | int | `42` | Random seed for reproducibility |

In [ ]:
!python scripts/move_random_images_colab.py \
    --images-dir input_files/raw_images \
    --labels-dir input_files/export/labels \
    --out        input_files/test-dataset \
    --count 20 \
    --seed 42